# Stellar-Origin Black Hole Binaries (SOBBHs) in BBHx

SOBBHs are inspiraling stellar-mass black-hole binaries that LISA observes years
before they leave the LISA band and become LIGO/Virgo merger triggers. From LISA's
perspective they are **inspiral-only** sources: slowly chirping, narrowband at any
given moment, no merger or ringdown in band. That makes their data-analysis
pipeline structurally closer to galactic-binary (GB) chunked-heterodyne than to
MBHB PhenomHM -- and that's exactly the design `bbhx` ships:

| | MBHB (PhenomHM) | SOBBH |
|---|---|---|
| Waveform model | IMR phenomenological | PN inspiral (TaylorT3) |
| Pipeline | `bbhx.waveforms.phenomhm` + heterodyne likelihood | `SOBBHTDIonTheFly` + chunked-heterodyne |
| Parameters | 11 (M, q, spins, t_c, ...) | 11 (m1, m2, s1, s2, dist, f_low, phi_c, inc, psi, lam, beta) |
| In-band duration | hours -- days | months -- years |
| Carrier behaviour | dominant chirp through merger | quasi-monochromatic, slow fdot |

This tutorial walks through:

1. Generating a SOBBH TD waveform with `lisatools.response.tdionfly.SOBBHTDIonTheFly`.
2. Transforming to the WDM domain.
3. The chunked-heterodyne likelihood via `bbhx.sobbhcomps.SOBBHWDMComputations`
   -- a thin BBHx-side sub-class of the shared `lisatools.chunked_het.WDMComputationsBase`
   that GB and SOBBH both build on. Same `fill_global_wdm` / `get_ll_wdm` /
   `get_swap_ll_wdm` / `get_ll_grad_wdm` surface as `GBWDMComputations`; only
   the SOBBH-specific routing constants differ.
4. A pointer at the JAX path (`bbhx.jax.sources.sobbh.JaxSOBBHSource`).

**Sprint architectural rule** (2026-06-06): GB-specific code lives in GBGPU,
SOBBH-specific code lives in BBHx, anything shared lives in lisatools.
Importing `SOBBHWDMComputations` from `gbgpu.gbcomps` no longer works --
use `from bbhx.sobbhcomps import SOBBHWDMComputations`.

**Prerequisites**:
[LISAanalysistools/examples/wdm_transform_tutorial.ipynb](../../LISAanalysistools/examples/wdm_transform_tutorial.ipynb)
(what the WDM domain is) and
[GBGPU/examples/fast_likelihood_tutorial.ipynb](../../GBGPU/examples/fast_likelihood_tutorial.ipynb)
(the chunked-heterodyne mechanics for GB sources -- everything carries over
verbatim with the rename `GBWDMComputations` -> `SOBBHWDMComputations`).

## Setup -- BBHx-native imports only

In [ ]:
import warnings
warnings.simplefilter('ignore', DeprecationWarning)

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# `import bbhx` registers the bbhx_<flavor> backend family used below.
import bbhx
from bbhx.jax.sources.sobbh import JaxSOBBHSource
from bbhx.sobbhcomps import SOBBHWDMComputations

from lisatools.detector import ESAOrbits
from lisatools.domains import TDSettings, TDSignal, WDMSettings, WDMSignal
from lisatools.sensitivity import XYZ2SensitivityMatrix
from lisatools.utils.constants import YRSID_SI

from lisatools.response.tdiconfig import TDIConfig
from lisatools.response.tdionfly import SOBBHTDIonTheFly

### WDM grid + observation window

Same canonical grid as the GB tutorial: `Nf=1460, Nt=2560` at `dt=10s`,
~1 year baseline.

In [ ]:
dt = 10.0
Nf, Nt = 1460, 2560
Nobs = Nf * Nt
t_start = int(0.5 * YRSID_SI / dt) * dt
Tobs = Nobs * dt
print(f'Nobs = {Nobs:,} samples  ({Tobs/YRSID_SI:.2f} yr)')

EC = 20  # edge-cut
wdm_set = WDMSettings(
    Nf, Nt, dt, t0=t_start,
    min_freq=1e-4, max_freq=35e-3,
    min_time=EC * Nf * dt, max_time=(Nt - EC) * Nf * dt,
)
Nf_active = int(wdm_set.ind_max_f - wdm_set.ind_min_f + 1)
Nt_active = int(wdm_set.Nt_active)
layer_df = wdm_set.layer_df
print(f'WDM active band: Nf_active={Nf_active}  Nt_active={Nt_active}  '
      f'layer_df={layer_df*1e3:.4f} mHz')

### Orbits, TDI config, t_ref

Same scaffolding as the GB tutorial. `t_ref` is the time at which the
SOBBH `f_low` parameter is specified.

In [ ]:
orbits = ESAOrbits()
tdi_config = TDIConfig('2nd generation')
t_ref = t_start

## 1. Generate a SOBBH waveform via `SOBBHTDIonTheFly`

`SOBBHTDIonTheFly` evaluates the SOBBH amplitude + phase point-wise from the
PN intrinsic-quantity expressions (TaylorT3). Parameter order:

* `m1`, `m2` -- component masses [solar masses]
* `s1`, `s2` -- aligned spins [dimensionless]
* `distance` -- luminosity distance [parsecs]
* `f_low` -- GW frequency at `t_ref` [Hz]
* `phi_c` -- reference orbital phase [rad]
* `inc`, `psi`, `lam`, `beta` -- inclination, polarization, ecliptic longitude/latitude [rad]

Pick a benchmark SOBBH at `f_low = 5 mHz` -- safely in LISA's bucket and
well below merger frequency.

In [ ]:
t_tdi = np.linspace(t_start, t_start + Tobs, 16384)

sobbh_gen = SOBBHTDIonTheFly(
    t_tdi, Tobs, t_ref, 1.0 / dt, 1,
    tdi_config=tdi_config, orbits=orbits, tdi_chan='XYZ',
)
print('SOBBHTDIonTheFly built. backend =', sobbh_gen.backend.name)

In [ ]:
m1_inj      = 30.0
m2_inj      = 25.0
s1_inj      = 0.1
s2_inj      = 0.0
distance_pc = 400e6        # 400 Mpc -> parsecs
f_low_inj   = 5.0e-3       # 5 mHz
phi_c_inj   = 1.0
inc_inj     = np.pi / 3.0
psi_inj     = 0.7
lam_inj     = 2.1
beta_inj    = 0.5

params_inj = np.array([
    m1_inj, m2_inj, s1_inj, s2_inj, distance_pc, f_low_inj,
    phi_c_inj, inc_inj, psi_inj, lam_inj, beta_inj,
])
print('SOBBH params (11):', params_inj.shape)

Generate the dense TD waveform on the full observation grid:

In [ ]:
t_arr = np.arange(Nobs) * dt + t_start
spline = sobbh_gen(
    *[np.array([p]) for p in params_inj],
    convert_to_ra_dec=False, return_spline=True,
)
td_inj = np.asarray(spline.eval_tdi(t_arr))[0]   # (nchannels, Nobs)
print('td_inj shape:', td_inj.shape, '  channel-0 peak |h|:',
      np.abs(td_inj[0]).max())

## 2. WDM transform of the SOBBH injection

Wrap each TD channel in a `TDSignal` and forward-transform to WDM. SOBBHs
show up as a **slowly drifting bright track** -- they live much longer in
band than MBHBs but evolve much faster than GBs.

In [ ]:
td_set = TDSettings(N=Nobs, dt=dt)
wdm_inj = TDSignal(td_inj, settings=td_set).transform(wdm_set)
wdm_arr = wdm_inj.arr[0]   # channel-0; shape (Nf_active, Nt_active)
print('wdm_inj.arr shape:', wdm_inj.arr.shape, '  -> using channel 0')

m_floor_active = int(np.floor(f_low_inj / layer_df)) - wdm_set.ind_min_f
print(f'f_low = {f_low_inj*1e3:.3f} mHz  -> active m_floor = {m_floor_active}')

## 3. Chunked-heterodyne likelihood via `SOBBHWDMComputations`

`SOBBHWDMComputations` is the SOBBH analog of `GBWDMComputations` --
same chunked-het pipeline, only the routing differs:

* `_BACKEND_PREFIX = "bbhx"` -- dispatches through `bbhx_<flavor>`.
* `_WRAP_ATTR = "SOBBHComputationGroupWrap"`.
* `_METHOD_PREFIX = "sobbh_wdm_het"`.
* `_NPARAMS = 11`, `_F0_PARAM_INDEX = 5`.

Everything else (`fill_global_wdm`, `get_ll_wdm`, gradient hooks,
layer-grouping, ...) is inherited from
`lisatools.chunked_het.WDMComputationsBase` and works identically to the GB
version.

In [ ]:
chunked_sobbh = SOBBHWDMComputations(
    wdm_set,
    t_ref=t_ref,
    Nt_sub=256,
    n_pad=32,
    N_sparse=256,
    N_cp_sig=0,           # direct path -> matches lisatools direct
    N_cp_orbit=0,
    orbits=orbits,
    tdi_config='2nd generation',
    d_d=0.0,              # source-only return
    tdi_type='XYZ',
)
print(f'backend           = {chunked_sobbh.backend.name}')
print(f'n_chunks          = {chunked_sobbh.n_chunks}')
print(f'T_chunk           = {chunked_sobbh.T_chunk:.2e} s')
print(f'tukey_alpha (auto)= {chunked_sobbh.resolved_tukey_alpha}')

### `fill_global_wdm` template build + mismatch check

Same call pattern as the GB version, with the 11-parameter SOBBH layout.
The active-band mismatch against the dense injection should land at the
lisatools-direct floor (`mm ~ 1e-9`).

In [ ]:
template_full = np.zeros((3, Nf, Nt), dtype=float)
chunked_sobbh.fill_global_wdm(
    params_inj.reshape(1, 11), template_full,
    convert_to_ra_dec=False, factors=None,
)

inj_active      = np.asarray(wdm_inj.arr)
template_active = template_full[:,
                                 wdm_set.ind_min_f:wdm_set.ind_max_f + 1,
                                 wdm_set.active_slice_t]
print('inj_active     :', inj_active.shape)
print('template_active:', template_active.shape)

ip_dd = float(np.sum(inj_active * inj_active))
ip_dh = float(np.sum(inj_active * template_active))
ip_hh = float(np.sum(template_active * template_active))
mm    = 1.0 - ip_dh / np.sqrt(ip_dd * ip_hh)
print(f'\nactive-band inner products (no PSD):')
print(f'  <d|d>     = {ip_dd:.4e}')
print(f'  <d|h>     = {ip_dh:.4e}')
print(f'  <h|h>     = {ip_hh:.4e}')
print(f'  mismatch  = {mm:+.3e}   (expect ~1e-9, lisatools-direct floor)')

## 4. JAX path: `JaxSOBBHSource`

For workflows that need `jax.grad` / `jax.jit` (NUTS samplers, custom
differentiable likelihood pipelines), `bbhx.jax.sources.sobbh.JaxSOBBHSource`
is a pure-JAX implementation of the SOBBH amplitude + phase. It matches the
C++ kernel at the inner-product level (`reldiff <= 1e-12`).

In [ ]:
print('JaxSOBBHSource:', JaxSOBBHSource)
doc = (JaxSOBBHSource.__doc__ or '').strip()
if doc:
    print('docstring (first 3 lines):')
    for line in doc.split('\n')[:3]:
        print(' ', line)

## Where to go next

* [bbhx_tutorial.ipynb](bbhx_tutorial.ipynb) -- the MBHB / PhenomHM pipeline
  in this repo. Different waveform model, but shares the LISA-response
  + heterodyne-likelihood infrastructure.
* [GBGPU/examples/fast_likelihood_tutorial.ipynb](../../GBGPU/examples/fast_likelihood_tutorial.ipynb)
  -- the GB version of this exact chunked-heterodyne pipeline. Same
  `WDMComputationsBase` parent, only the routing constants differ.
* [LISAanalysistools/examples/wdm_transform_tutorial.ipynb](../../LISAanalysistools/examples/wdm_transform_tutorial.ipynb)
  -- WDM domain primer.